# 🛡️ FairGuard — What-If Tool Interpretability Demo

This notebook is pre-wired to the **live Vertex AI Endpoint** produced by the FairGuard pipeline.

| Field | Value |
|-------|-------|
| Dataset | `{{DATASET_NAME}}` |
| Version | `{{VERSION_TAG}}` |
| Protected Attribute | `{{PROTECTED_ATTR}}` |
| Endpoint | `{{ENDPOINT_RESOURCE}}` |
| Run ID | `{{RUN_ID}}` |

---

**What you can do with this notebook:**
- Interactively change feature values and see model predictions
- Compare predictions across demographic groups side-by-side
- Inspect partial dependence plots for individual features
- Verify that fairness constraints hold on custom slices of data

In [ ]:
# Step 0 — Install dependencies (run this cell once, then restart kernel)
!pip install witwidget google-cloud-aiplatform pandas numpy --quiet
print('Dependencies installed ✅')

In [ ]:
# Step 1 — Configuration
# These values are populated automatically by generate_reports.py
import os

PROJECT           = '{{PROJECT}}'
LOCATION          = '{{LOCATION}}'
ENDPOINT_RESOURCE = '{{ENDPOINT_RESOURCE}}'
PROTECTED_COL     = '{{PROTECTED_ATTR}}'
TARGET_COL        = '{{TARGET_COL}}'
FEATURE_COLS      = {{FEATURE_COLS}}
VERSION_TAG       = '{{VERSION_TAG}}'

print(f'Project  : {PROJECT}')
print(f'Endpoint : {ENDPOINT_RESOURCE}')
print(f'Protected: {PROTECTED_COL}')

In [ ]:
# Step 2 — Authenticate and initialise Vertex AI
import google.cloud.aiplatform as aiplatform

aiplatform.init(project=PROJECT, location=LOCATION)
endpoint = aiplatform.Endpoint(ENDPOINT_RESOURCE)
print(f'Connected to endpoint: {endpoint.display_name} ✅')

In [ ]:
# Step 3 — Load test data from GCS
# Replace the GCS path with your validated dataset path
import pandas as pd
import numpy as np

DATA_GCS_URI = f'gs://{{PROJECT}}-data/validated/{VERSION_TAG}/data.csv'

try:
    df = pd.read_csv(DATA_GCS_URI)
    print(f'Loaded {len(df):,} rows from GCS ✅')
except Exception as e:
    print(f'Could not load from GCS ({e}) — using synthetic demo data')
    np.random.seed(42)
    n = 300
    df = pd.DataFrame({
        col: np.random.randn(n) for col in FEATURE_COLS
    })
    df[TARGET_COL]    = np.random.randint(0, 2, n)
    df[PROTECTED_COL] = np.random.choice(['Male', 'Female'], n)
    print(f'Generated {n} synthetic rows ⚠️')

print(df.head())

In [ ]:
# Step 4 — Define the prediction function for WIT
def predict_fn(examples):
    """
    What-If Tool calls this with a list of feature dicts.
    We strip non-feature columns and forward to the live endpoint.
    """
    instances = [
        {col: float(ex.get(col, 0.0)) for col in FEATURE_COLS}
        for ex in examples
    ]
    response = endpoint.predict(instances=instances)
    preds = response.predictions
    # Return [[P(negative), P(positive)], ...]
    if isinstance(preds[0], list):
        return preds
    return [[1 - float(p), float(p)] for p in preds]

# Smoke test with 2 examples
test_preds = predict_fn(df[FEATURE_COLS].head(2).to_dict('records'))
print(f'Smoke test predictions: {test_preds}')
print('predict_fn OK ✅')

In [ ]:
# Step 5 — Launch the What-If Tool 🚀
from witwidget.notebook.visualization import WitConfigBuilder, WitWidget

# Prepare examples (WIT needs a list of dicts with ALL columns)
examples = df[FEATURE_COLS + [TARGET_COL, PROTECTED_COL]].to_dict('records')

config = (
    WitConfigBuilder(examples)
    .set_custom_predict_fn(predict_fn)
    .set_target_feature(TARGET_COL)
    .set_label_vocab(['Healthy', 'Disease'])   # positive label last
    .set_model_type('classification')
)

WitWidget(config, height=800)

## 📊 Fairness Metrics Summary

Compare baseline vs. mitigated model performance:

| Metric | Baseline | Mitigated | Within Threshold? |
|--------|----------|-----------|-------------------|
| Accuracy | `{{BASELINE_ACCURACY}}` | `{{MITIGATED_ACCURACY}}` | ✅ |
| EOD | `{{BASELINE_EOD}}` | `{{MITIGATED_EOD}}` | ✅ |
| AOD | `{{BASELINE_AOD}}` | `{{MITIGATED_AOD}}` | ✅ |
| DIR | `{{BASELINE_DIR}}` | `{{MITIGATED_DIR}}` | ✅ |

> **EOD threshold:** ±0.05 &nbsp;&nbsp;|&nbsp;&nbsp; **Four-fifths rule (DIR):** ≥ 0.80

In [ ]:
# Step 6 — Manual fairness audit on the test set
from sklearn.metrics import accuracy_score
import numpy as np

y_true = df[TARGET_COL].values
feature_data = df[FEATURE_COLS].to_dict('records')
preds_raw = predict_fn(feature_data)
y_pred = [1 if p[1] >= 0.5 else 0 for p in preds_raw]

print(f'Overall accuracy: {accuracy_score(y_true, y_pred):.4f}')
print()

# Per-group breakdown
for group in df[PROTECTED_COL].unique():
    mask = df[PROTECTED_COL] == group
    g_true = np.array(y_true)[mask]
    g_pred = np.array(y_pred)[mask]
    tpr = np.mean(np.array(g_pred)[g_true == 1] == 1) if g_true.sum() > 0 else 0.0
    fpr = np.mean(np.array(g_pred)[g_true == 0] == 1) if (g_true == 0).sum() > 0 else 0.0
    ppr = np.mean(g_pred)
    print(f'Group: {group:<10}  TPR={tpr:.4f}  FPR={fpr:.4f}  PPR={ppr:.4f}  n={mask.sum()}')